## **DEPURACION DEFUNCIONES 2021**


**1. IMPORTAR LIBRERIAS**

In [27]:
### IMPORTAR LIBRERIAS ###

!pip install pycountry
!pip install xlrd==2.0.1
!pip install openpyxl
!pip install pyarrow --upgrade
!pip install fastparquet

import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import pycountry
import warnings
warnings.filterwarnings('ignore')

**1.2. IMPORTAR CAUSAS DE DEFUNCIÓN**

In [2]:
def leer_lista_105(ruta_archivo):
    """
    Lee el archivo de la Lista 105 de Colombia para tabulación de mortalidad.
    
    Parámetros:
    -----------
    ruta_archivo : str
        Ruta al archivo Excel (ej: "data/raw/Referenciales/Lista_105_Colombia_CIE9-y-CIE10.xls")
    
    Retorna:
    --------
    pd.DataFrame con columnas:
        - No_Lista: Número de la lista
        - Causa: Descripción de la causa
        - Codigos_CIE10: Códigos CIE-10
        - Codigos_CIE9: Códigos CIE-9
    """
    
    print(f"Leyendo archivo: {ruta_archivo}")
    
    # Leer el archivo Excel
    df = pd.read_excel(ruta_archivo, dtype=str)
    
    # Mostrar columnas detectadas
    print(f"Columnas detectadas: {list(df.columns)}")
    
    # Buscar la fila donde comienzan los datos (después del encabezado "LISTA COLOMBIA 105...")
    inicio_datos = None
    for idx, row in df.iterrows():
        if 'No. Lista' in str(row.values) or 'No.Lista' in str(row.values):
            inicio_datos = idx
            break
    
    if inicio_datos is None:
        # Intentar lectura alternativa
        print("Buscando encabezados de forma alternativa...")
        df = pd.read_excel(ruta_archivo, header=None, dtype=str)
        
        for idx, row in df.iterrows():
            row_str = ' '.join([str(x) for x in row.values if pd.notna(x)])
            if 'No. Lista' in row_str or 'No.Lista' in row_str:
                inicio_datos = idx
                break
    
    if inicio_datos is not None:
        print(f"Datos encontrados a partir de la fila: {inicio_datos}")
        # Leer desde la fila de inicio
        df = pd.read_excel(ruta_archivo, header=inicio_datos, dtype=str)
    
    # Limpiar nombres de columnas
    df.columns = df.columns.astype(str).str.strip()
    
    # Identificar las columnas correctas
    col_numero = None
    col_causa = None
    col_cie10 = None
    col_cie9 = None
    
    for col in df.columns:
        col_lower = col.lower()
        if 'no' in col_lower and 'lista' in col_lower:
            col_numero = col
        elif 'causa' in col_lower:
            col_causa = col
        elif 'cie-10' in col_lower or 'cie10' in col_lower:
            col_cie10 = col
        elif 'cie-9' in col_lower or 'cie9' in col_lower:
            col_cie9 = col
    
    print(f"Columnas identificadas:")
    print(f"  - Número: {col_numero}")
    print(f"  - Causa: {col_causa}")
    print(f"  - CIE-10: {col_cie10}")
    print(f"  - CIE-9: {col_cie9}")
    
    # Crear DataFrame normalizado
    df_limpio = pd.DataFrame()
    
    if all([col_numero, col_causa, col_cie10, col_cie9]):
        df_limpio['No_Lista'] = df[col_numero]
        df_limpio['Causa'] = df[col_causa]
        df_limpio['Codigos_CIE10'] = df[col_cie10]
        df_limpio['Codigos_CIE9'] = df[col_cie9]
    else:
        # Intento por posición de columnas
        print("Usando posición de columnas (A, B, C, D)")
        df_limpio['No_Lista'] = df.iloc[:, 0]
        df_limpio['Causa'] = df.iloc[:, 1]
        df_limpio['Codigos_CIE10'] = df.iloc[:, 2]
        df_limpio['Codigos_CIE9'] = df.iloc[:, 3]
    
    # Eliminar filas vacías
    df_limpio = df_limpio.dropna(subset=['No_Lista'], how='all')
    df_limpio = df_limpio[df_limpio['No_Lista'].notna()]
    
    # Limpiar espacios
    for col in df_limpio.columns:
        df_limpio[col] = df_limpio[col].astype(str).str.strip()
    
    # Eliminar filas que no tienen número de lista válido
    df_limpio = df_limpio[df_limpio['No_Lista'].str.isdigit()]
    
    # Convertir número de lista a entero
    df_limpio['No_Lista'] = df_limpio['No_Lista'].astype(int)
    
    print(f"\n✅ Archivo procesado exitosamente")
    print(f"   Total de causas: {len(df_limpio)}")
    print(f"   Rango de listas: {df_limpio['No_Lista'].min()} - {df_limpio['No_Lista'].max()}")
    
    return df_limpio

**1.3. IMPORTAR CÓDIGOS DAVIPOLA**

In [3]:
##############################################################
# Leer DIVIPOLA (códigos de departamentos y municipios) #
##############################################################

print("\n2. LEYENDO DIVIPOLA...")
divipola = pd.read_csv("data/raw/Referenciales/DIVIPOLA_CentrosPoblados.csv",
                       encoding='latin-1', sep=';', dtype=str)

print(f"   Columnas disponibles: {list(divipola.columns)}")

# Identificar las columnas correctas (pueden tener nombres diferentes)
# Buscar columnas que contengan las palabras clave
col_cod_mun = [c for c in divipola.columns if 'Municipio' in c and 'digo' in c][0]
col_nom_dpto = [c for c in divipola.columns if 'Departamento' in c and 'Nombre' in c][0]
col_nom_mun = [c for c in divipola.columns if 'Municipio' in c and 'Nombre' in c][0]

print(f"   Usando columnas: {col_cod_mun}, {col_nom_dpto}, {col_nom_mun}")

# Extraer códigos de departamento y municipio del código de 5 dígitos
divipola['COD_DPTO'] = divipola[col_cod_mun].str[:2]
divipola['COD_MUNIC'] = divipola[col_cod_mun].str[2:]

divipola_merge = divipola[['COD_DPTO', 'COD_MUNIC', col_nom_dpto, col_nom_mun]].copy()
divipola_merge.columns = ['COD_DPTO', 'COD_MUNIC', 'NOMBRE_DEPARTAMENTO', 'NOMBRE_MUNICIPIO']

# Eliminar duplicados
divipola_merge = divipola_merge.drop_duplicates(subset=['COD_DPTO', 'COD_MUNIC'])

print(f"   Registros únicos: {len(divipola_merge):,}")
print(f"   Muestra:")
print(divipola_merge.head(3)[['COD_DPTO', 'COD_MUNIC', 'NOMBRE_DEPARTAMENTO', 'NOMBRE_MUNICIPIO']])


2. LEYENDO DIVIPOLA...
   Columnas disponibles: ['Código_Departamento', 'Nombre_Departamento', 'Código_Municipio', 'Nombre_Municipio', 'Código_Entidad', 'Nombre_Entidad', 'Tipo', 'Longitud', 'Latitud']
   Usando columnas: Código_Municipio, Nombre_Departamento, Nombre_Municipio
   Registros únicos: 1,122
   Muestra:
   COD_DPTO COD_MUNIC NOMBRE_DEPARTAMENTO NOMBRE_MUNICIPIO
0        05       001           ANTIOQUIA         MEDELLÍN
28       05       002           ANTIOQUIA        ABEJORRAL
32       05       004           ANTIOQUIA         ABRIAQUÍ


**1.4. IMPORTAR LISTA 6/66 Y 6/67**

In [4]:
def cargar_comparativo(
    path="data/raw/Referenciales/Comparativo-lista-agrupada-666-basada-lista667-OMS-OPS-vs-lista667-OMS-OPS.xlsx"
):
    """
    Carga el archivo Excel especificando los tipos de dato directamente.
    """
    # Leer con dtype específico para las columnas de Grupo
    df = pd.read_excel(
        path, 
        sheet_name="Lista 667 vs 666", 
        header=8,
        dtype={0: str, 3: str}  # Columnas 0 y 3 como string desde el inicio
    )
    
    # Seleccionar columnas
    cols = [0, 1, 2, 3, 4, 5, -1]
    df = df.iloc[:, cols]

    df.columns = [
        "Grupo_666", "Descripcion_666", "Codigos_CIE10_666",
        "Grupo_667", "Descripcion_667", "Codigos_CIE10_667",
        "Diferencias"
    ]

    df.dropna(how='all', inplace=True)
    
    # Limpiar: si viene como "100.0" lo convierte a "100"
    df['Grupo_666'] = df['Grupo_666'].str.replace('.0', '', regex=False).str.strip()
    df['Grupo_667'] = df['Grupo_667'].str.replace('.0', '', regex=False).str.strip()
    
    # Eliminar vacíos
    df = df[df['Grupo_666'].notna() & (df['Grupo_666'] != '')]
    df = df[df['Grupo_667'].notna() & (df['Grupo_667'] != '')]

    # Eliminar duplicados
    df = df.drop_duplicates(subset=["Grupo_666"])
    df = df.drop_duplicates(subset=["Grupo_667"])

    return df

**2. CREAR FUNCIONES**


**2.1. FUNCIÓN LLAMADO DE DATOS**

In [5]:
def leer_archivos_defunciones_csv(ruta_carpeta, año_inicio, año_fin):
    """Lee y unifica archivos CSV en un rango de años, manejando distintos formatos de codificación."""
    ruta = Path(ruta_carpeta)
    archivos = [
        arch for arch in ruta.glob("Defun*.csv")
        if any(str(año) in arch.stem for año in range(año_inicio, año_fin + 1))
    ]

    if not archivos:
        print(" No se encontraron archivos CSV en el rango indicado.")
        return pd.DataFrame()

    dfs = []
    print(f"Leyendo archivos {año_inicio}-{año_fin}:")
    for archivo in tqdm(archivos):
        cargado = False
        for encoding in ["utf-8", "latin-1", "windows-1252"]:
            try:
                df = pd.read_csv(archivo, encoding=encoding, sep=',', low_memory=False)
                dfs.append(df)
                print(f" Cargado: {archivo.name} con codificación {encoding} ({len(df):,} filas)")
                cargado = True
                break
            except Exception as e:
                continue
        if not cargado:
            print(f" Error al leer {archivo.name}: no se pudo decodificar con utf-8, latin-1 ni windows-1252")

    if dfs:
        df_total = pd.concat(dfs, ignore_index=True)
    else:
        df_total = pd.DataFrame()

    return df_total

**2.2. FUNCIÓN MAPEO DESCRIPCIONES**

In [6]:
def crear_mapeo_descripciones(iso_standard='3166-1', language='es'):
    """
    Crea diccionarios de mapeo para las descripciones
    
    Parámetros:
    -----------
    iso_standard : str, opcional (default='3166-1')
        Estándar ISO a utilizar para países:
        - '3166-1': Códigos de países (recomendado para CODPRES)
        - '3166-2': Códigos de subdivisiones (estados/provincias)
    
    language : str, opcional (default='es')
        Idioma para los nombres de países:
        - 'es': Español
        - 'en': Inglés
        - 'fr': Francés, etc.
        Requiere babel instalado para idiomas distintos a inglés
    
    Retorna:
    --------
    dict : Diccionario con todos los mapeos de códigos
    
    Ejemplos:
    ---------
    >>> mapeos = crear_mapeo_descripciones()  # Por defecto ISO 3166-1
    >>> mapeos = crear_mapeo_descripciones(iso_standard='3166-2')
    >>> mapeos = crear_mapeo_descripciones(language='en')
    """
    
    # Generar rangos para PERMAN_MUN (01-98 años)
    permanencia_municipio = {
        'X1': 'MENOS DE 1 DÍA',
        'X2': 'DE 1 A 6 DÍAS',
        'X3': 'DE 7 A 29 DÍAS',
        'X4': 'DE 1 A 5 MESES',
        'X5': 'DE 6 A 11 MESES',
        '00': 'TIEMPO DESCONOCIDO',
        '99': 'DE 99 Y MÁS AÑOS'
    }

    # Agregar dinámicamente 01-98
    for i in range(1, 99):
        clave = f"{i:02d}"
        años = "AÑO" if i == 1 else "AÑOS"
        permanencia_municipio[clave] = f"{i} {años}"

    # Número de hijos vivos
    n_hijosv = {}
    for i in range(0, 21):
        texto = "HIJO NACIDO VIVO" if i == 1 else "HIJOS NACIDOS VIVOS"
        clave_padded = f"{i:02d}"
        clave_simple = str(i)
        valor = f"{i} {texto}"
        n_hijosv[clave_padded] = valor
        n_hijosv[clave_simple] = valor
    n_hijosv['99'] = 'SIN INFORMACIÓN'

    # Número de hijos muertos
    n_hijosm = {}
    for i in range(1, 16):
        texto = "HIJO NACIDO MUERTO" if i == 1 else "HIJOS NACIDOS MUERTOS"
        clave_padded = f"{i:02d}"
        clave_simple = str(i)
        valor = f"{i} {texto}"
        n_hijosm[clave_padded] = valor
        n_hijosm[clave_simple] = valor
    n_hijosm['99'] = 'SIN INFORMACIÓN'

    # ===========================================================================
    # MAPEO DE PAÍSES (CODPRES) usando pycountry
    # ===========================================================================
    codpres_iso = {
        # Códigos especiales de DANE
        '998': 'RESTO DE PAÍSES',
        '999': 'SIN INFORMACIÓN'
    }
    
    try:
        if iso_standard == '3166-1':
            # Usar códigos de países (ISO 3166-1 numérico)
            for country in pycountry.countries:
                code = str(country.numeric).zfill(3)  # Asegurar 3 dígitos
                
                # Intentar traducir si babel está disponible y idioma != 'en'
                if language != 'en':
                    try:
                        from babel import Locale
                        locale = Locale(language)
                        translated_name = locale.territories.get(country.alpha_2, country.name)
                        codpres_iso[code] = translated_name.upper()
                    except (ImportError, Exception):
                        # Si babel no está disponible, usar nombre en inglés
                        codpres_iso[code] = country.name.upper()
                else:
                    codpres_iso[code] = country.name.upper()
                    
        elif iso_standard == '3166-2':
            # Usar subdivisiones (ISO 3166-2) - útil para provincias/estados
            print("⚠️  Advertencia: ISO 3166-2 no usa códigos numéricos.")
            print("   Se recomienda usar iso_standard='3166-1' para CODPRES.")
            
            # Igualmente cargar países como fallback
            for country in pycountry.countries:
                code = str(country.numeric).zfill(3)
                codpres_iso[code] = country.name.upper()
        else:
            raise ValueError(f"iso_standard debe ser '3166-1' o '3166-2', no '{iso_standard}'")
            
        print(f"✓ Países cargados usando ISO {iso_standard} ({len(codpres_iso)-2} países)")
        
    except ImportError:
        print("⚠️  Advertencia: pycountry no está instalado.")
        print("   Instalar con: pip install pycountry")
        print("   Instalar babel para traducciones: pip install babel")
        print("   Usando mapeo limitado de países.")
        
        # Mapeo mínimo como fallback
        codpres_iso.update({
            '032': 'ARGENTINA',
            '068': 'BOLIVIA',
            '076': 'BRASIL',
            '124': 'CANADÁ',
            '152': 'CHILE',
            '170': 'COLOMBIA',
            '188': 'COSTA RICA',
            '192': 'CUBA',
            '214': 'REPÚBLICA DOMINICANA',
            '218': 'ECUADOR',
            '222': 'EL SALVADOR',
            '250': 'FRANCIA',
            '276': 'ALEMANIA',
            '320': 'GUATEMALA',
            '340': 'HONDURAS',
            '380': 'ITALIA',
            '484': 'MÉXICO',
            '558': 'NICARAGUA',
            '591': 'PANAMÁ',
            '600': 'PARAGUAY',
            '604': 'PERÚ',
            '724': 'ESPAÑA',
            '840': 'ESTADOS UNIDOS',
            '858': 'URUGUAY',
            '862': 'VENEZUELA',
        })
        
    except Exception as e:
        print(f"⚠️  Error al cargar países: {e}")

    # ===========================================================================
    # RETORNAR TODOS LOS MAPEOS
    # ===========================================================================
    return {
        'A_DEFUN': {
            '1': 'CABECERA MUNICIPAL',
            '2': 'CENTRO POBLADO',
            '3': 'RURAL DISPERSO',
            '9': 'SIN INFORMACIÓN'
        },
        'SEXO': {
            '1': 'MASCULINO',
            '2': 'FEMENINO',
            '3': 'INDETERMINADO'
        },
        'EST_CIVIL': {
            '1': 'NO ESTABA CASADO(A) Y LLEVABA DOS O MAS AÑOS VIVIENDO CON SU PAREJA',
            '2': 'NO ESTABA CASADO(A) Y LLEVABA MENOS DE DOS AÑOS VIVIENDO CON SU PAREJA',
            '3': 'ESTABA SEPARADO(A), DIVORCIADO(A)',
            '4': 'ESTABA VIUDO(A)',
            '5': 'ESTABA SOLTERO(A)',
            '6': 'ESTABA CASADO(A)',
            '9': 'SIN INFORMACIÓN'
        },
        'SIT_DEFUN': {
            '1': 'HOSPITAL/CLÍNICA',
            '2': 'CENTRO/PUESTO DE SALUD',
            '3': 'CASA/DOMICILIO',
            '4': 'LUGAR DE TRABAJO',
            '5': 'VÍA PÚBLICA',
            '6': 'OTRO',
            '9': 'SIN INFORMACIÓN'
        },
        'GRU_ED1': {
            '00': 'MENOR DE UNA HORA',
            '01': 'MENOR DE UN DÍA', '02': 'DE 1 A 6 DÍAS',
            '03': 'DE 7 A 27 DÍAS', '04': 'DE 28 A 29 DÍAS',
            '05': 'DE 1 A 5 MESES',
            '06': 'DE 6 A 11 MESES', '07': 'DE UN AÑO',
            '08': 'DE 2 A 4 AÑOS', '09': 'DE 5 A 9 AÑOS',
            '10': 'DE 10 A 14 AÑOS', '11': 'DE 15 A 19 AÑOS',
            '12': 'DE 20 A 24 AÑOS', '13': 'DE 25 A 29 AÑOS',
            '14': 'DE 30 A 34 AÑOS', '15': 'DE 35 A 39 AÑOS',
            '16': 'DE 40 A 44 AÑOS', '17': 'DE 45 A 49 AÑOS',
            '18': 'DE 50 A 54 AÑOS', '19': 'DE 55 A 59 AÑOS',
            '20': 'DE 60 A 64 AÑOS', '21': 'DE 65 A 69 AÑOS',
            '22': 'DE 70 A 74 AÑOS', '23': 'DE 75 A 79 AÑOS',
            '24': 'DE 80 A 84 AÑOS', '25': 'DE 85 A 89 AÑOS',
            '26': 'DE 90 A 94 AÑOS', '27': 'DE 95 A 99 AÑOS',
            '28': 'DE 100 AÑOS Y MÁS', '29': 'EDAD DESCONOCIDA'
        },
        'GRU_ED2': {
            '01': 'MENOR DE UN AÑO', '02': 'DE 1 A 4 AÑOS',
            '03': 'DE 5 A 14 AÑOS', '04': 'DE 15 A 44 AÑOS',
            '05': 'DE 45 A 64 AÑOS', '06': 'DE 65 Y MÁS AÑOS',
            '07': 'EDAD DESCONOCIDA'
        },
        'CONS_EXP': {
            '1': 'MÉDICO TRATANTE',
            '2': 'MÉDICO NO TRATANTE',
            '3': 'MÉDICO LEGISTA',
            '4': 'PERSONAL DE SALUD AUTORIZADO',
            '5': 'FUNCIONARIO DEL REGISTRO CIVIL',
            '6': 'OTRO',
            '9': 'SIN INFORMACIÓN'
        },
        'AREA_RES': {
            '1': 'CABECERA MUNICIPAL',
            '2': 'CENTRO POBLADO',
            '3': 'RURAL DISPERSO',
            '9': 'SIN INFORMACIÓN'
        },
        'P_PMAN_IRIS': {
            '1': 'NATURAL',
            '2': 'VIOLENTA',
            '3': 'EN ESTUDIO'
        },
        'TIPO_DEFUN': {
            '1': 'DEFUNCIÓN FETAL',
            '2': 'DEFUNCIÓN NO FETAL'
        },
        'NIVEL_EDU': {
            '1': 'PREESCOLAR',
            '2': 'BÁSICA PRIMARIA',
            '3': 'BÁSICA SECUNDARIA',
            '4': 'MEDIA ACADÉMICA O CLÁSICA',
            '5': 'MEDIA TÉCNICA',
            '6': 'NORMALISTA',
            '7': 'TÉCNICA PROFESIONAL',
            '8': 'TECNOLÓGICA',
            '9': 'PROFESIONAL',
            '10': 'ESPECIALIZACIÓN',
            '11': 'MAESTRÍA',
            '12': 'DOCTORADO',
            '13': 'NINGUNO',
            '99': 'SIN INFORMACIÓN'
        },
        'CODPRES': codpres_iso,
        'CODPTORE': {
            '05': 'ANTIOQUIA',
            '08': 'ATLÁNTICO',
            '11': 'BOGOTÁ',
            '13': 'BOLÍVAR',
            '15': 'BOYACÁ',
            '17': 'CALDAS',
            '18': 'CAQUETÁ',
            '19': 'CAUCA',
            '20': 'CESAR',
            '23': 'CÓRDOBA',
            '25': 'CUNDINAMARCA',
            '27': 'CHOCÓ',
            '41': 'HUILA',
            '44': 'LA GUAJIRA',
            '47': 'MAGDALENA',
            '50': 'META',
            '52': 'NARIÑO',
            '54': 'NORTE DE SANTANDER',
            '63': 'QUINDÍO',
            '66': 'RISARALDA',
            '68': 'SANTANDER',
            '70': 'SUCRE',
            '73': 'TOLIMA',
            '76': 'VALLE DEL CAUCA',
            '81': 'ARAUCA',
            '85': 'CASANARE',
            '86': 'PUTUMAYO',
            '88': 'ARCHIPIÉLAGO DE SAN ANDRÉS, PROVIDENCIA Y SANTA CATALINA',
            '91': 'AMAZONAS',
            '94': 'GUAINÍA',
            '95': 'GUAVIARE',
            '97': 'VAUPÉS',
            '99': 'VICHADA'
        },
        'SEG_SOCIAL': {
            '1': 'CONTRIBUTIVO',
            '2': 'SUBSIDIADO',
            '3': 'EXCEPCIÓN',
            '4': 'ESPECIAL',
            '5': 'NO ASEGURADO',
            '9': 'SIN INFORMACIÓN'
        },
        'MU_PARTO': {
            '1': 'ANTES',
            '2': 'DURANTE',
            '3': 'DESPUÉS',
            '4': 'IGNORADO',
            '9': 'SIN INFORMACIÓN'
        },
        'T_PARTO': {
            '1': 'ESPONTÁNEO',
            '2': 'CESÁREA',
            '3': 'INSTRUMENTADO',
            '4': 'IGNORADO',
            '9': 'SIN INFORMACIÓN'
        },
        'TIPO_EMB': {
            '1': 'SIMPLE',
            '2': 'DOBLE',
            '3': 'TRIPLE',
            '4': 'CUÁDRUPLE O MÁS',
            '5': 'IGNORADO',
            '9': 'SIN INFORMACIÓN'
        },
        'T_GES': {
            '1': 'MENOS DE 22',
            '2': 'DE 22 A 27',
            '3': 'DE 28 A 37',
            '4': 'DE 38 A 41',
            '5': 'DE 42 Y MÁS',
            '6': 'IGNORADO',
            '9': 'SIN INFORMACIÓN'
        },
        'PESO_NAC': {
            '1': 'MENOS DE 1000 GRAMOS',
            '2': 'DE 1000 A 1499 GRAMOS',
            '3': 'DE 1500 A 1999 GRAMOS',
            '4': 'DE 2000 A 2499 GRAMOS',
            '5': 'DE 2500 A 2999 GRAMOS',
            '6': 'DE 3000 A 3499 GRAMOS',
            '7': 'DE 3500 A 3999 GRAMOS',
            '8': 'DE 4000 Y MÁS GRAMOS',
            '9': 'SIN INFORMACIÓN'
        },
        'EDAD_MADRE': {
            '1': 'DE 10 A 14 AÑOS',
            '2': 'DE 15 A 19 AÑOS',
            '3': 'DE 20 A 24 AÑOS',
            '4': 'DE 25 A 29 AÑOS',
            '5': 'DE 30 A 34 AÑOS',
            '6': 'DE 35 A 39 AÑOS',
            '7': 'DE 40 A 44 AÑOS',
            '8': 'DE 45 A 49 AÑOS',
            '9': 'DE 50 A 54 AÑOS',
            '99': 'SIN INFORMACIÓN'
        },
        'EST_CIVM': {
            '1': 'NO ESTABA CASADO(A) Y LLEVABA DOS O MAS AÑOS VIVIENDO CON SU PAREJA',
            '2': 'NO ESTABA CASADO(A) Y LLEVABA MENOS DE DOS AÑOS VIVIENDO CON SU PAREJA',
            '3': 'ESTABA SEPARADO(A), DIVORCIADO(A)',
            '4': 'ESTABA VIUDO(A)',
            '5': 'ESTABA SOLTERO(A)',
            '6': 'ESTABA CASADO(A)',
            '9': 'SIN INFORMACIÓN'
        },
        'NIV_EDUM': {
            '1': 'PREESCOLAR',
            '2': 'BÁSICA PRIMARIA',
            '3': 'BÁSICA SECUNDARIA',
            '4': 'MEDIA ACADÉMICA O CLÁSICA',
            '5': 'MEDIA TÉCNICA',
            '6': 'NORMALISTA',
            '7': 'TÉCNICA PROFESIONAL',
            '8': 'TECNOLÓGICA',
            '9': 'PROFESIONAL',
            '10': 'ESPECIALIZACIÓN',
            '11': 'MAESTRÍA',
            '12': 'DOCTORADO',
            '13': 'NINGUNO',
            '99': 'SIN INFORMACIÓN'
        },
        'EMB_FAL': {
            '1': 'SI',
            '2': 'NO',
            '9': 'SIN INFORMACIÓN'
        },
        'EMB_SEM': {
            '1': 'SI',
            '2': 'NO',
            '9': 'SIN INFORMACIÓN'
        },
        'EMB_MES': {
            '1': 'SI',
            '2': 'NO',
            '9': 'SIN INFORMACIÓN'
        },
        'MAN_MUER': {
            '1': 'SUICIDIO',
            '2': 'HOMICIDIO',
            '3': 'ACCIDENTE DE TRÁNSITO',
            '4': 'OTRO ACCIDENTE',
            '5': 'EN ESTUDIO',
            '9': 'SIN INFORMACIÓN'
        },
        'CODOCUR': {
            '05': 'ANTIOQUIA',
            '08': 'ATLÁNTICO',
            '11': 'BOGOTÁ',
            '13': 'BOLÍVAR',
            '15': 'BOYACÁ',
            '17': 'CALDAS',
            '18': 'CAQUETÁ',
            '19': 'CAUCA',
            '20': 'CESAR',
            '23': 'CÓRDOBA',
            '25': 'CUNDINAMARCA',
            '27': 'CHOCÓ',
            '41': 'HUILA',
            '44': 'LA GUAJIRA',
            '47': 'MAGDALENA',
            '50': 'META',
            '52': 'NARIÑO',
            '54': 'NORTE DE SANTANDER',
            '63': 'QUINDÍO',
            '66': 'RISARALDA',
            '68': 'SANTANDER',
            '70': 'SUCRE',
            '73': 'TOLIMA',
            '76': 'VALLE DEL CAUCA',
            '81': 'ARAUCA',
            '85': 'CASANARE',
            '86': 'PUTUMAYO',
            '88': 'ARCHIPIÉLAGO DE SAN ANDRÉS, PROVIDENCIA Y SANTA CATALINA',
            '91': 'AMAZONAS',
            '94': 'GUAINÍA',
            '95': 'GUAVIARE',
            '97': 'VAUPÉS',
            '99': 'VICHADA'
        },
        'C_MUERTE': {
            '1': 'NECROPSIA',
            '2': 'HISTORIA CLÍNICA',
            '3': 'PRUEBAS DE LABORATORIO',
            '4': 'INTERROGATORIO A FAMILIARES O TESTIGOS',
            '9': 'SIN INFORMACIÓN'
        },
        'ASIS_MED': {
            '1': 'SÍ',
            '2': 'NO',
            '3': 'IGNORADO',
            '9': 'SIN INFORMACIÓN'
        },
        'IDADMISALUD': {
            '1': 'ENTIDAD PROMOTORA DE SALUD',
            '2': 'ENTIDAD PROMOTORA DE SALUD - SUBSIDIADO',
            '3': 'ENTIDAD ADAPTADA DE SALUD',
            '4': 'ENTIDAD ESPECIAL DE SALUD',
            '5': 'ENTIDAD EXCEPTUADA DE SALUD',
            '9': 'SIN INFORMACIÓN'
        },
        'T_GES_AGRU_CIE': {
            '1': 'MENOS DE 22 SEMANAS',
            '2': 'DE 22 A 27 SEMANAS',
            '3': 'DE 28 A 36 SEMANAS',
            '4': 'DE 37 A 41 SEMANAS',
            '5': 'DE 42 Y MÁS SEMANAS',
            '6': 'IGNORADO',
            '9': 'SIN INFORMACIÓN'
        },
        'IDPROFCER': {
            '1': 'MÉDICO',
            '2': 'ENFERMERO(A)',
            '3': 'AUXILIAR DE ENFERMERÍA',
            '4': 'PROMOTOR(A) DE SALUD',
            '5': 'FUNCIONARIO DE REGISTRO CIVIL',
            '6': 'MÉDICO LEGISTA',
            '9': 'SIN INFORMACIÓN'
        },
        'N_HIJOSV': n_hijosv,
        'N_HIJOSM': n_hijosm,
        'TIEM_PER': permanencia_municipio
    }

**2.3. FUNCIÓN HOMOLOGAR DEPARTAMENTOS DE RECIDENCIA**

In [7]:
def homologar_departamentos_residencia(df, divipola):
    """Homologa departamentos y municipios de residencia"""
    # Preparar códigos
    df['CODPTORE'] = pd.to_numeric(df['CODPTORE'], errors='coerce').astype('Int64')
    df['CODMUNRE'] = pd.to_numeric(df['CODMUNRE'], errors='coerce').astype('Int64')

    df['CODPTORE'] = df['CODPTORE'].astype(str).str.replace('<NA>', 'nan').str.zfill(2)
    df['CODMUNRE'] = df['CODMUNRE'].astype(str).str.replace('<NA>', 'nan').str.zfill(3)
    
    # Corregir código antiguo 83 -> 18 (Caquetá)
    df.loc[df['CODPTORE'] == '83', 'CODPTORE'] = '18'
    
    # Asegurar que no hay duplicados en divipola
    divipola_unico = divipola.drop_duplicates(subset=['COD_DPTO', 'COD_MUNIC'])
    
    # Merge principal
    df = df.merge(
        divipola_unico[['COD_DPTO', 'COD_MUNIC', 'NOMBRE_DEPARTAMENTO', 'NOMBRE_MUNICIPIO']],
        left_on=['CODPTORE', 'CODMUNRE'],
        right_on=['COD_DPTO', 'COD_MUNIC'],
        how='left',
        suffixes=('', '_RESIDENCIA')
    )
    
    print(f"   Registros después del merge: {len(df):,}")
    
    # Completar nulos con cabeceras municipales
    cabeceras = divipola_unico[divipola_unico['COD_MUNIC'] == '001'][
        ['COD_DPTO', 'NOMBRE_DEPARTAMENTO', 'NOMBRE_MUNICIPIO']
    ].drop_duplicates(subset=['COD_DPTO']).rename(columns={
        'NOMBRE_DEPARTAMENTO': 'DEPT_CAB',
        'NOMBRE_MUNICIPIO': 'MUNIC_CAB'
    })
    
    df = df.merge(cabeceras, left_on='CODPTORE', right_on='COD_DPTO', 
                  how='left', suffixes=('', '_y'))
    
    print(f"   Registros después de cabeceras: {len(df):,}")
    
    mask = df['NOMBRE_DEPARTAMENTO_RESIDENCIA'].isna()
    df.loc[mask, 'NOMBRE_DEPARTAMENTO_RESIDENCIA'] = df.loc[mask, 'DEPT_CAB']
    df.loc[mask, 'NOMBRE_MUNICIPIO_RESIDENCIA'] = df.loc[mask, 'MUNIC_CAB']
    
    # Marcar los que no se encontraron
    df['NOMBRE_DEPARTAMENTO_RESIDENCIA'].fillna('NO REGISTRA', inplace=True)
    df['NOMBRE_MUNICIPIO_RESIDENCIA'].fillna('NO REGISTRA', inplace=True)
    
    # Limpiar columnas temporales
    cols_drop = ['DEPT_CAB', 'MUNIC_CAB', 'COD_DPTO_y', 'COD_DPTO_RESIDENCIA', 
                 'COD_MUNIC_RESIDENCIA']
    df.drop([c for c in cols_drop if c in df.columns], axis=1, inplace=True)
    
    return df

**2.4. FUNCIÓN BUSCAR CIE 10**

In [8]:
def buscar_cie10_optimizado(c_bas1_val, cie_dict):
    """
    Búsqueda flexible de códigos CIE-10 en un diccionario.
    Soporta variaciones como:
    - Código exacto
    - Código sin punto (ej: A003 → A00.3)
    - Código solo con 3 caracteres (ej: A00)
    - Código numérico que podría requerir prefijo (ej: 003 → A00)
    """
    if pd.isna(c_bas1_val):
        return None

    codigo = str(c_bas1_val).strip().upper()

    # 1. ---- BÚSQUEDA EXACTA ----
    if codigo in cie_dict:
        return cie_dict[codigo]

    # 2. ---- ELIMINAR PUNTOS Y REINTENTAR ----
    codigo_sin_punto = codigo.replace(".", "")
    if codigo_sin_punto in cie_dict:
        return cie_dict[codigo_sin_punto]

    # 3. ---- RECONSTRUIR FORMATO CIE-10 (LETRA + 2-3 DÍGITOS) ----
    # ej: A003 → A00.3
    if len(codigo_sin_punto) >= 4:
        reconstruido = codigo_sin_punto[0] + codigo_sin_punto[1:3] + "." + codigo_sin_punto[3:]
        if reconstruido in cie_dict:
            return cie_dict[reconstruido]

    # 4. ---- SOLO 3 CARACTERES (ej: A00, F32, J18) ----
    if len(codigo_sin_punto) == 3:
        if codigo_sin_punto in cie_dict:
            return cie_dict[codigo_sin_punto]

    # 5. ---- SI ES SOLO NUMÉRICO → AGREGAR TODAS LAS LETRAS POSIBLES ----
    # Ej: usuario pone "003" → probar A00, B00, C00, ...
    if codigo.isdigit() and len(codigo) >= 2:
        for letra in "ABCDEFGHIJKLMNOPQRSTUVWXYZ":
            candidato = letra + codigo.zfill(2)[:2]  # Ej 03 →  A03
            if candidato in cie_dict:
                return cie_dict[candidato]

    # 6. ---- ÚLTIMO RECURSO: buscar por prefijo de 3 caracteres ----
    prefijo3 = codigo[:3].replace(".", "")
    if prefijo3 in cie_dict:
        return cie_dict[prefijo3]

    return None

**2.5. FUNCIÓN EXPANDIR CÓDIGO CIE**

In [9]:
def expandir_codigos_cie(df_lista105, columna_codigo='Codigos_CIE9'):
    """
    Expande los rangos de códigos CIE a códigos individuales.
    
    Ejemplo: "001-009" se expande a ["001", "002", ..., "009"]
    
    Parámetros:
    -----------
    df_lista105 : pd.DataFrame
        DataFrame con la Lista 105
    columna_codigo : str
        Nombre de la columna con los códigos ('Codigos_CIE9' o 'Codigos_CIE10')
    
    Retorna:
    --------
    pd.DataFrame con una fila por cada código individual
    """
    
    registros = []
    
    for _, row in df_lista105.iterrows():
        codigos_str = str(row[columna_codigo])
        
        if pd.isna(codigos_str) or codigos_str == 'nan':
            continue
        
        # Separar por comas
        codigos = [c.strip() for c in codigos_str.split(',')]
        
        for codigo in codigos:
            # Si es un rango (ej: "001-009")
            if '-' in codigo:
                partes = codigo.split('-')
                if len(partes) == 2:
                    inicio, fin = partes
                    inicio = inicio.strip()
                    fin = fin.strip()
                    
                    # Intentar expandir el rango
                    try:
                        # Extraer la parte numérica
                        prefijo_inicio = ''.join([c for c in inicio if not c.isdigit()])
                        num_inicio = int(''.join([c for c in inicio if c.isdigit()]))
                        
                        prefijo_fin = ''.join([c for c in fin if not c.isdigit()])
                        num_fin = int(''.join([c for c in fin if c.isdigit()]))
                        
                        # Generar códigos en el rango
                        for num in range(num_inicio, num_fin + 1):
                            codigo_expandido = f"{prefijo_inicio}{str(num).zfill(len(str(num_inicio)))}"
                            registros.append({
                                'No_Lista': row['No_Lista'],
                                'Causa': row['Causa'],
                                'Codigo': codigo_expandido
                            })
                    except:
                        # Si falla, agregar el código completo sin expandir
                        registros.append({
                            'No_Lista': row['No_Lista'],
                            'Causa': row['Causa'],
                            'Codigo': codigo
                        })
            else:
                # Código individual
                registros.append({
                    'No_Lista': row['No_Lista'],
                    'Causa': row['Causa'],
                    'Codigo': codigo
                })
    
    df_expandido = pd.DataFrame(registros)
    
    print(f"\n✅ Códigos expandidos: {len(df_expandido)} registros")
    
    return df_expandido

**2.6. FUNCIÓN HOMMOLOGAR CAUSA DE DEFUNNCIÓN CIE**

In [10]:
def homologar_causa_lista105(df_defunciones, df_lista105, col_codigo='CAU_HOMOL', 
                              crear_columna='CAU_HOMOL_DESC'):
    """
    Homologa el campo CAU_HOMOL con la Lista 105 de Colombia.
    
    Parámetros:
    -----------
    df_defunciones : pd.DataFrame
        DataFrame con las defunciones que tiene el campo CAU_HOMOL
    df_lista105 : pd.DataFrame
        DataFrame con la Lista 105 (resultado de leer_lista_105)
    col_codigo : str
        Nombre de la columna con el código de la lista 105 en df_defunciones (default: 'CAU_HOMOL')
    crear_columna : str
        Nombre de la columna a crear con la descripción (default: 'CAU_HOMOL_DESC')
    
    Retorna:
    --------
    pd.DataFrame con columna adicional:
        - {crear_columna}: Descripción de la causa según lista 105 (campo 'Causa')
    """
    
    print(f"\nHomologando CAU_HOMOL con Lista 105 de Colombia...")
    print(f"   Campo a cruzar: {col_codigo}")
    print(f"   Campo a crear: {crear_columna}")
    
    # Preparar datos para el merge
    # Asegurar que ambos campos sean string y estén limpios
    df_defunciones[col_codigo] = df_defunciones[col_codigo].astype(str).str.strip()
    df_lista105['No_Lista'] = df_lista105['No_Lista'].astype(str).str.strip()
    
    # Crear diccionario de mapeo para mayor eficiencia
    dict_lista105 = dict(zip(df_lista105['No_Lista'], df_lista105['Causa']))
    
    print(f"   Total códigos en Lista 105: {len(dict_lista105)}")
    print(f"   Ejemplo de mapeo: {list(dict_lista105.items())[:3]}")
    
    # Primera búsqueda: mapeo directo
    print(f"\n   Paso 1: Búsqueda directa...")
    df_defunciones[crear_columna] = df_defunciones[col_codigo].map(dict_lista105)
    
    coincidencias_directas = df_defunciones[crear_columna].notna().sum()
    print(f"   Coincidencias directas: {coincidencias_directas:,}")
    
    # Segunda búsqueda: quitar ceros a la izquierda para los no encontrados
    sin_coinc = df_defunciones[crear_columna].isna()
    if sin_coinc.sum() > 0:
        print(f"\n   Paso 2: Búsqueda sin ceros iniciales para {sin_coinc.sum():,} registros...")
        
        # Crear versión sin ceros a la izquierda
        df_defunciones.loc[sin_coinc, 'CAU_HOMOL_TEMP'] = (
            df_defunciones.loc[sin_coinc, col_codigo]
            .str.lstrip('0')  # Quitar ceros a la izquierda
        )
        
        # Aplicar mapeo con códigos sin ceros
        df_defunciones.loc[sin_coinc, crear_columna] = (
            df_defunciones.loc[sin_coinc, 'CAU_HOMOL_TEMP'].map(dict_lista105)
        )
        
        # Eliminar columna temporal
        df_defunciones.drop('CAU_HOMOL_TEMP', axis=1, inplace=True, errors='ignore')
        
        nuevas_coinc = df_defunciones.loc[sin_coinc, crear_columna].notna().sum()
        print(f"   Nuevas coincidencias: {nuevas_coinc:,}")
    
    # Estadísticas finales
    coincidencias_totales = df_defunciones[crear_columna].notna().sum()
    porcentaje = (coincidencias_totales / len(df_defunciones)) * 100
    
    print(f"\n✅ Homologación completada:")
    print(f"   Total registros: {len(df_defunciones):,}")
    print(f"   Coincidencias totales: {coincidencias_totales:,} ({porcentaje:.2f}%)")
    print(f"   Sin coincidencia: {(len(df_defunciones) - coincidencias_totales):,}")
    
    # Mostrar algunos valores únicos de CAU_HOMOL sin coincidencia
    sin_coinc_final = df_defunciones[df_defunciones[crear_columna].isna()][col_codigo].unique()
    if len(sin_coinc_final) > 0:
        print(f"\n   Ejemplos de códigos sin coincidencia final: {sin_coinc_final[:5]}")
    
    return df_defunciones

**2.7. FUNCIÓN ALMACENAR RESULTADOS**

In [11]:
def homologar_causa_directa(df_defunciones, df_cie10, col_codigo='C_DIR1', crear_columna='C_DIR1_DESC'):
    """
    Homologa el campo C_DIR1 (Causa Directa) con CIE-10.
    
    Parámetros:
    -----------
    df_defunciones : pd.DataFrame
        DataFrame con las defunciones
    df_cie10 : pd.DataFrame
        DataFrame con códigos CIE-10 y sus descripciones (columnas: 'CIE10', 'LITERAL10')
    col_codigo : str
        Nombre de la columna con el código (default: 'C_DIR1')
    crear_columna : str
        Nombre de la columna a crear (default: 'C_DIR1_DESC')
    
    Retorna:
    --------
    pd.DataFrame con columna adicional con la descripción
    """
    
    print(f"\n📋 Homologando {col_codigo} (Causa Directa - renglón a)...")
    print(f"   Campo a cruzar: {col_codigo}")
    print(f"   Campo a crear: {crear_columna}")
    
    # Preparar datos
    df_defunciones[col_codigo] = df_defunciones[col_codigo].astype(str).str.strip().str.upper()
    df_cie10['CIE10'] = df_cie10['CIE10'].astype(str).str.strip().str.upper()
    
    # Eliminar duplicados en CIE-10
    df_cie10_unico = df_cie10.drop_duplicates(subset=['CIE10'])
    
    # Crear diccionario de mapeo
    dict_cie10 = dict(zip(df_cie10_unico['CIE10'], df_cie10_unico['LITERAL10']))
    
    print(f"   Total códigos CIE-10: {len(dict_cie10):,}")
    
    # Mapeo directo
    df_defunciones[crear_columna] = df_defunciones[col_codigo].map(dict_cie10)
    
    # Estadísticas
    coincidencias = df_defunciones[crear_columna].notna().sum()
    total = df_defunciones[col_codigo].notna().sum()
    porcentaje = (coincidencias / total * 100) if total > 0 else 0
    
    print(f"   ✅ Coincidencias: {coincidencias:,} de {total:,} ({porcentaje:.2f}%)")
    print(f"   ⚠️  Sin coincidencia: {(total - coincidencias):,}")
    
    # Mostrar ejemplos sin coincidencia
    sin_coinc = df_defunciones[
        df_defunciones[col_codigo].notna() & 
        df_defunciones[crear_columna].isna()
    ][col_codigo].unique()
    
    if len(sin_coinc) > 0:
        print(f"   Ejemplos sin match: {list(sin_coinc[:5])}")
    
    return df_defunciones

**2.8. FUNCIÓN HOMOLOGAR CAUSA DIRECTA**

In [12]:
def homologar_causa_directa(df_defunciones, df_cie10, col_codigo='C_DIR1', crear_columna='C_DIR1_DESC'):
    """
    Homologa el campo C_DIR1 (Causa Directa) con CIE-10.
    
    Parámetros:
    -----------
    df_defunciones : pd.DataFrame
        DataFrame con las defunciones
    df_cie10 : pd.DataFrame
        DataFrame con códigos CIE-10 y sus descripciones (columnas: 'CIE10', 'LITERAL10')
    col_codigo : str
        Nombre de la columna con el código (default: 'C_DIR1')
    crear_columna : str
        Nombre de la columna a crear (default: 'C_DIR1_DESC')
    
    Retorna:
    --------
    pd.DataFrame con columna adicional con la descripción
    """
    
    print(f"\n📋 Homologando {col_codigo} (Causa Directa - renglón a)...")
    print(f"   Campo a cruzar: {col_codigo}")
    print(f"   Campo a crear: {crear_columna}")
    
    # Preparar datos
    df_defunciones[col_codigo] = df_defunciones[col_codigo].astype(str).str.strip().str.upper()
    df_cie10['CIE10'] = df_cie10['CIE10'].astype(str).str.strip().str.upper()
    
    # Eliminar duplicados en CIE-10
    df_cie10_unico = df_cie10.drop_duplicates(subset=['CIE10'])
    
    # Crear diccionario de mapeo
    dict_cie10 = dict(zip(df_cie10_unico['CIE10'], df_cie10_unico['LITERAL10']))
    
    print(f"   Total códigos CIE-10: {len(dict_cie10):,}")
    
    # Mapeo directo
    df_defunciones[crear_columna] = df_defunciones[col_codigo].map(dict_cie10)
    
    # Estadísticas
    coincidencias = df_defunciones[crear_columna].notna().sum()
    total = df_defunciones[col_codigo].notna().sum()
    porcentaje = (coincidencias / total * 100) if total > 0 else 0
    
    print(f"   ✅ Coincidencias: {coincidencias:,} de {total:,} ({porcentaje:.2f}%)")
    print(f"   ⚠️  Sin coincidencia: {(total - coincidencias):,}")
    
    # Mostrar ejemplos sin coincidencia
    sin_coinc = df_defunciones[
        df_defunciones[col_codigo].notna() & 
        df_defunciones[crear_columna].isna()
    ][col_codigo].unique()
    
    if len(sin_coinc) > 0:
        print(f"   Ejemplos sin match: {list(sin_coinc[:5])}")
    
    return df_defunciones

**2.9. FUNCIÓN HOMOLOGAR CAUSA ANTECEDENTE 1**

In [13]:
def homologar_causa_antecedente1(df_defunciones, df_cie10, col_codigo='CAUSA_MULT', crear_columna='C_ANT1_DESC'):
    """
    Homologa el campo CAUSA_MULT (Causa Antecedente 1) con CIE-10.
    
    Parámetros:
    -----------
    df_defunciones : pd.DataFrame
        DataFrame con las defunciones
    df_cie10 : pd.DataFrame
        DataFrame con códigos CIE-10 y sus descripciones
    col_codigo : str
        Nombre de la columna con el código (default: 'CAUSA_MULT')
    crear_columna : str
        Nombre de la columna a crear (default: 'C_ANT1_DESC')
    
    Retorna:
    --------
    pd.DataFrame con columna adicional con la descripción
    """
    
    print(f"\n📋 Homologando {col_codigo} (Causa Antecedente 1 - renglón b)...")
    print(f"   Campo a cruzar: {col_codigo}")
    print(f"   Campo a crear: {crear_columna}")
    
    # Preparar datos
    df_defunciones[col_codigo] = df_defunciones[col_codigo].astype(str).str.strip().str.upper()
    df_cie10['CIE10'] = df_cie10['CIE10'].astype(str).str.strip().str.upper()
    
    # Eliminar duplicados
    df_cie10_unico = df_cie10.drop_duplicates(subset=['CIE10'])
    dict_cie10 = dict(zip(df_cie10_unico['CIE10'], df_cie10_unico['LITERAL10']))
    
    print(f"   Total códigos CIE-10: {len(dict_cie10):,}")
    
    # Mapeo
    df_defunciones[crear_columna] = df_defunciones[col_codigo].map(dict_cie10)
    
    # Estadísticas
    coincidencias = df_defunciones[crear_columna].notna().sum()
    total = df_defunciones[col_codigo].notna().sum()
    porcentaje = (coincidencias / total * 100) if total > 0 else 0
    
    print(f"   ✅ Coincidencias: {coincidencias:,} de {total:,} ({porcentaje:.2f}%)")
    print(f"   ⚠️  Sin coincidencia: {(total - coincidencias):,}")
    
    return df_defunciones

**2.10. FUNCIÓN HOMOLOGAR CAUSA ANTECEDENTE 2**

In [14]:
def homologar_causa_antecedente2(df_defunciones, df_cie10, col_codigo='C_ANT2', crear_columna='C_ANT2_DESC'):
    """
    Homologa el campo C_ANT2 (Causa Antecedente 2) con CIE-10.
    
    Parámetros:
    -----------
    df_defunciones : pd.DataFrame
        DataFrame con las defunciones
    df_cie10 : pd.DataFrame
        DataFrame con códigos CIE-10 y sus descripciones
    col_codigo : str
        Nombre de la columna con el código (default: 'C_ANT2')
    crear_columna : str
        Nombre de la columna a crear (default: 'C_ANT2_DESC')
    
    Retorna:
    --------
    pd.DataFrame con columna adicional con la descripción
    """
    
    print(f"\n📋 Homologando {col_codigo} (Causa Antecedente 2 - renglón c)...")
    print(f"   Campo a cruzar: {col_codigo}")
    print(f"   Campo a crear: {crear_columna}")
    
    # Preparar datos
    df_defunciones[col_codigo] = df_defunciones[col_codigo].astype(str).str.strip().str.upper()
    df_cie10['CIE10'] = df_cie10['CIE10'].astype(str).str.strip().str.upper()
    
    # Eliminar duplicados
    df_cie10_unico = df_cie10.drop_duplicates(subset=['CIE10'])
    dict_cie10 = dict(zip(df_cie10_unico['CIE10'], df_cie10_unico['LITERAL10']))
    
    print(f"   Total códigos CIE-10: {len(dict_cie10):,}")
    
    # Mapeo
    df_defunciones[crear_columna] = df_defunciones[col_codigo].map(dict_cie10)
    
    # Estadísticas
    coincidencias = df_defunciones[crear_columna].notna().sum()
    total = df_defunciones[col_codigo].notna().sum()
    porcentaje = (coincidencias / total * 100) if total > 0 else 0
    
    print(f"   ✅ Coincidencias: {coincidencias:,} de {total:,} ({porcentaje:.2f}%)")
    print(f"   ⚠️  Sin coincidencia: {(total - coincidencias):,}")
    
    return df_defunciones

**2.11. FUNCIÓN HOMOLOGAR CAUSA ANTECEDENTE 3**

In [15]:
def homologar_causa_antecedente3(df_defunciones, df_cie10, col_codigo='C_ANT3', crear_columna='C_ANT3_DESC'):
    """
    Homologa el campo C_ANT3 (Causa Antecedente 3) con CIE-10.
    
    Parámetros:
    -----------
    df_defunciones : pd.DataFrame
        DataFrame con las defunciones
    df_cie10 : pd.DataFrame
        DataFrame con códigos CIE-10 y sus descripciones
    col_codigo : str
        Nombre de la columna con el código (default: 'C_ANT3')
    crear_columna : str
        Nombre de la columna a crear (default: 'C_ANT3_DESC')
    
    Retorna:
    --------
    pd.DataFrame con columna adicional con la descripción
    """
    
    print(f"\n📋 Homologando {col_codigo} (Causa Antecedente 3 - renglón d)...")
    print(f"   Campo a cruzar: {col_codigo}")
    print(f"   Campo a crear: {crear_columna}")
    
    # Preparar datos
    df_defunciones[col_codigo] = df_defunciones[col_codigo].astype(str).str.strip().str.upper()
    df_cie10['CIE10'] = df_cie10['CIE10'].astype(str).str.strip().str.upper()
    
    # Eliminar duplicados
    df_cie10_unico = df_cie10.drop_duplicates(subset=['CIE10'])
    dict_cie10 = dict(zip(df_cie10_unico['CIE10'], df_cie10_unico['LITERAL10']))
    
    print(f"   Total códigos CIE-10: {len(dict_cie10):,}")
    
    # Mapeo
    df_defunciones[crear_columna] = df_defunciones[col_codigo].map(dict_cie10)
    
    # Estadísticas
    coincidencias = df_defunciones[crear_columna].notna().sum()
    total = df_defunciones[col_codigo].notna().sum()
    porcentaje = (coincidencias / total * 100) if total > 0 else 0
    
    print(f"   ✅ Coincidencias: {coincidencias:,} de {total:,} ({porcentaje:.2f}%)")
    print(f"   ⚠️  Sin coincidencia: {(total - coincidencias):,}")
    
    return df_defunciones

**2.12. FUNCIÓN HOMOLOGAR OTRAS PATOLOGÍAS**

In [16]:
def homologar_otros_patologicos(df_defunciones, df_cie10, col_codigo='C_PAT1', crear_columna='C_PAT1_DESC'):
    """
    Homologa el campo C_PAT1 (Otros Estados Patológicos Importantes) con CIE-10.
    
    Parámetros:
    -----------
    df_defunciones : pd.DataFrame
        DataFrame con las defunciones
    df_cie10 : pd.DataFrame
        DataFrame con códigos CIE-10 y sus descripciones
    col_codigo : str
        Nombre de la columna con el código (default: 'C_PAT1')
    crear_columna : str
        Nombre de la columna a crear (default: 'C_PAT1_DESC')
    
    Retorna:
    --------
    pd.DataFrame con columna adicional con la descripción
    """
    
    print(f"\n📋 Homologando {col_codigo} (Otros Estados Patológicos - sección II)...")
    print(f"   Campo a cruzar: {col_codigo}")
    print(f"   Campo a crear: {crear_columna}")
    
    # Preparar datos
    df_defunciones[col_codigo] = df_defunciones[col_codigo].astype(str).str.strip().str.upper()
    df_cie10['CIE10'] = df_cie10['CIE10'].astype(str).str.strip().str.upper()
    
    # Eliminar duplicados
    df_cie10_unico = df_cie10.drop_duplicates(subset=['CIE10'])
    dict_cie10 = dict(zip(df_cie10_unico['CIE10'], df_cie10_unico['LITERAL10']))
    
    print(f"   Total códigos CIE-10: {len(dict_cie10):,}")
    
    # Mapeo
    df_defunciones[crear_columna] = df_defunciones[col_codigo].map(dict_cie10)
    
    # Estadísticas
    coincidencias = df_defunciones[crear_columna].notna().sum()
    total = df_defunciones[col_codigo].notna().sum()
    porcentaje = (coincidencias / total * 100) if total > 0 else 0
    
    print(f"   ✅ Coincidencias: {coincidencias:,} de {total:,} ({porcentaje:.2f}%)")
    print(f"   ⚠️  Sin coincidencia: {(total - coincidencias):,}")
    
    return df_defunciones

**2.13. FUNCIÓN HOMOLOGAR MUERTE SIN CERTIFICACIÓN MÉDICA**

In [17]:
def homologar_muerte_sin_certificacion(df_defunciones, df_cie10, col_codigo='C_MCM1', crear_columna='C_MCM1_DESC'):
    """
    Homologa el campo C_MCM1 (Muerte Sin Certificación Médica) con CIE-10.
    
    Parámetros:
    -----------
    df_defunciones : pd.DataFrame
        DataFrame con las defunciones
    df_cie10 : pd.DataFrame
        DataFrame con códigos CIE-10 y sus descripciones
    col_codigo : str
        Nombre de la columna con el código (default: 'C_MCM1')
    crear_columna : str
        Nombre de la columna a crear (default: 'C_MCM1_DESC')
    
    Retorna:
    --------
    pd.DataFrame con columna adicional con la descripción
    """
    
    print(f"\n📋 Homologando {col_codigo} (Muerte Sin Certificación Médica)...")
    print(f"   Campo a cruzar: {col_codigo}")
    print(f"   Campo a crear: {crear_columna}")
    
    # Preparar datos
    df_defunciones[col_codigo] = df_defunciones[col_codigo].astype(str).str.strip().str.upper()
    df_cie10['CIE10'] = df_cie10['CIE10'].astype(str).str.strip().str.upper()
    
    # Eliminar duplicados
    df_cie10_unico = df_cie10.drop_duplicates(subset=['CIE10'])
    dict_cie10 = dict(zip(df_cie10_unico['CIE10'], df_cie10_unico['LITERAL10']))
    
    print(f"   Total códigos CIE-10: {len(dict_cie10):,}")
    
    # Mapeo
    df_defunciones[crear_columna] = df_defunciones[col_codigo].map(dict_cie10)
    
    # Estadísticas
    coincidencias = df_defunciones[crear_columna].notna().sum()
    total = df_defunciones[col_codigo].notna().sum()
    porcentaje = (coincidencias / total * 100) if total > 0 else 0
    
    print(f"   ✅ Coincidencias: {coincidencias:,} de {total:,} ({porcentaje:.2f}%)")
    print(f"   ⚠️  Sin coincidencia: {(total - coincidencias):,}")
    
    return df_defunciones

**2.14. FUNCIÓN HOMOLOGAR CAUSA 666**

In [18]:
def anexar_desc(df_defun, df_ref, columna_origen, lista_referencia):
    """
    Cruza df_defun con df_ref de forma flexible.
    
    Parámetros:
    -----------
    df_defun : pd.DataFrame
        DataFrame de defunciones que contiene la columna a homologar
    df_ref : pd.DataFrame
        DataFrame de referencia con las descripciones (resultado de cargar_comparativo)
    columna_origen : str
        Nombre exacto de la columna en df_defun a homologar (ej: 'CAUSA_666', 'CAU_HOMOL')
    lista_referencia : str
        Lista a usar en df_ref: "666" o "667"
        - "666": Busca en Grupo_666 y Descripcion_666
        - "667": Busca en Grupo_667 y Descripcion_667
    
    Retorna:
    --------
    pd.DataFrame con columna adicional {columna_origen}_DESC
    
    Ejemplos:
    ---------
    # Homologar CAUSA_666 usando lista 666
    df = anexar_desc(df, df_ref, columna_origen='CAUSA_666', lista_referencia='666')
    
    # Homologar CAU_HOMOL usando lista 667
    df = anexar_desc(df, df_ref, columna_origen='CAU_HOMOL', lista_referencia='667')
    """
    
    # Validación de inputs
    assert lista_referencia in ["666", "667"], "lista_referencia debe ser '666' o '667'"
    
    if columna_origen not in df_defun.columns:
        raise KeyError(f"El dataframe no contiene la columna '{columna_origen}'")
    
    # Definir nombres dinámicos según la lista de referencia
    grupo_col = f"Grupo_{lista_referencia}"
    desc_col_ref = f"Descripcion_{lista_referencia}"
    desc_col_new = f"{columna_origen}_DESC"
    
    # Verificar que las columnas existen en df_ref
    if grupo_col not in df_ref.columns:
        raise KeyError(f"El dataframe de referencia no contiene '{grupo_col}'")
    if desc_col_ref not in df_ref.columns:
        raise KeyError(f"El dataframe de referencia no contiene '{desc_col_ref}'")
    
    # Preparar datos para merge (asegurar tipo string y limpiar espacios)
    df_defun[columna_origen] = df_defun[columna_origen].astype(str).str.strip()
    df_ref[grupo_col] = df_ref[grupo_col].astype(str).str.strip()
    
    # Hacer merge
    df_merged = df_defun.merge(
        df_ref[[grupo_col, desc_col_ref]],
        left_on=columna_origen,
        right_on=grupo_col,
        how="left"
    )
    
    # Renombrar la columna de descripción
    df_merged = df_merged.rename(columns={desc_col_ref: desc_col_new})
    
    # Remover columna auxiliar del merge
    df_merged = df_merged.drop(columns=[grupo_col])
    
    return df_merged

**2.15. FUNCIÓN PARA ALMACENAR RESULTADOS**

In [19]:
def guardar_dataframe_procesado(df, nombre_archivo, ruta_carpeta="data/processed/"):
    """
    Guarda el DataFrame procesado en formato Parquet (óptimo para pandas).
    Si el archivo existe, lo reemplaza.
    
    Parámetros:
    -----------
    df : pd.DataFrame
        DataFrame a guardar
    nombre_archivo : str
        Nombre del archivo sin extensión (ej: 'defunciones_1979_1991')
    ruta_carpeta : str
        Ruta de la carpeta donde guardar (default: 'data/processed/')
    
    Retorna:
    --------
    str: Ruta completa del archivo guardado
    """
    from pathlib import Path
    import os
    
    # Crear carpeta si no existe
    Path(ruta_carpeta).mkdir(parents=True, exist_ok=True)
    
    # Construir ruta completa con extensión .parquet
    archivo_parquet = Path(ruta_carpeta) / f"{nombre_archivo}.parquet"
    
    # Verificar si existe
    if archivo_parquet.exists():
        print(f"   ⚠️  El archivo ya existe: {archivo_parquet}")
        print(f"   📝 Reemplazando con nueva versión...")
        # Eliminar archivo anterior
        os.remove(archivo_parquet)
    else:
        print(f"   ✨ Creando nuevo archivo: {archivo_parquet}")
    
    # Guardar en formato Parquet (más eficiente que CSV para pandas)
    df.to_parquet(archivo_parquet, engine="fastparquet", index=False, compression='snappy')
    
    # Obtener tamaño del archivo
    tamaño_mb = archivo_parquet.stat().st_size / (1024 * 1024)
    
    print(f"   ✅ Archivo guardado exitosamente")
    print(f"   📊 Registros: {len(df):,}")
    print(f"   📁 Tamaño: {tamaño_mb:.2f} MB")
    print(f"   📂 Ruta: {archivo_parquet}")
    
    # Mostrar cómo leerlo después
    print(f"\n   💡 Para leer este archivo después, usa:")
    print(f"      df = pd.read_parquet('{archivo_parquet}')")
    
    return str(archivo_parquet)

**3. EJECUTAR PROCESO PRINCIPAL**


In [20]:
# =============================================================================
#PROCESO PRINCIPAL
# =============================================================================

print("=" * 60)
print("ANÁLISIS EXPLORATORIO DE DATOS - DEFUNCIONES 2021")
print("=" * 60)

# 2.1. Leer archivos de defunciones
print("\n1. LEYENDO ARCHIVOS DE DEFUNCIONES...")
df_defun = leer_archivos_defunciones_csv("data/raw/Muertes", 2021,2021)
print(f"   Total registros: {len(df_defun):,}")
print(f"   Columnas: {list(df_defun.columns)}")

ANÁLISIS EXPLORATORIO DE DATOS - DEFUNCIONES 2021

1. LEYENDO ARCHIVOS DE DEFUNCIONES...
Leyendo archivos 2021-2021:


100%|██████████| 1/1 [00:01<00:00,  1.68s/it]

 Cargado: Defun2021.csv con codificación latin-1 (363,089 filas)
   Total registros: 363,089
   Columnas: ['COD_DPTO', 'COD_MUNIC', 'A_DEFUN', 'SIT_DEFUN', 'OTRSITIODE', 'TIPO_DEFUN', 'ANO', 'MES', 'HORA', 'MINUTOS', 'SEXO', 'EST_CIVIL', 'GRU_ED1', 'GRU_ED2', 'NIVEL_EDU', 'ULTCURFAL', 'MUERTEPORO', 'SIMUERTEPO', 'OCUPACION', 'IDPERTET', 'CODPRES', 'CODPTORE', 'CODMUNRE', 'AREA_RES', 'SEG_SOCIAL', 'IDADMISALUD', 'P_PMAN_IRIS', 'CONS_EXP', 'MU_PARTO', 'T_PARTO', 'TIPO_EMB', 'T_GES', 'T_GES_AGRU_CIE', 'PESO_NAC', 'EDAD_MADRE', 'N_HIJOSV', 'N_HIJOSM', 'EST_CIVM', 'NIV_EDUM', 'ULTCURMAD', 'EMB_FAL', 'EMB_SEM', 'EMB_MES', 'CODOCUR', 'CODMUNOC', 'C_MUERTE', 'C_MUERTEB', 'C_MUERTEC', 'C_MUERTED', 'C_MUERTEE', 'ASIS_MED', 'CAUSA_MULT', 'C_BAS1', 'CAUSA_667', 'IDPROFCER', 'CAU_HOMOL']


In [21]:
# Número total de columnas_1
num_columnas = df_defun.shape[1]
print("Número de columnas:", num_columnas)

display(df_defun.head())

list(df_defun.columns)



Número de columnas: 56


,COD_DPTO,COD_MUNIC,A_DEFUN,SIT_DEFUN,OTRSITIODE,TIPO_DEFUN,ANO,MES,HORA,MINUTOS,...,C_MUERTEB,C_MUERTEC,C_MUERTED,C_MUERTEE,ASIS_MED,CAUSA_MULT,C_BAS1,CAUSA_667,IDPROFCER,CAU_HOMOL
0,54,1,1,1,NaN,2,2021,5,3.0,25.0,...,1.0,NaN,NaN,NaN,1,J960/J129/U071*E149,U071,109,1,10
1,54,1,1,1,NaN,2,2021,12,6.0,5.0,...,1.0,NaN,NaN,NaN,1,R570/I269/I742*I10,I742,309,1,58
2,54,1,1,1,NaN,2,2021,7,19.0,30.0,...,1.0,NaN,NaN,NaN,1,J969/G934 N19/N179/T659 X49*F319,X499,509,1,99
3,63,1,1,3,NaN,2,2021,2,21.0,30.0,...,1.0,NaN,1.0,NaN,1,I509/R688/R590,I509,306,1,54
4,66,682,1,3,NaN,2,2021,2,2.0,0.0,...,NaN,NaN,1.0,NaN,2,I219,I219,303,1,51


['COD_DPTO',
 'COD_MUNIC',
 'A_DEFUN',
 'SIT_DEFUN',
 'OTRSITIODE',
 'TIPO_DEFUN',
 'ANO',
 'MES',
 'HORA',
 'MINUTOS',
 'SEXO',
 'EST_CIVIL',
 'GRU_ED1',
 'GRU_ED2',
 'NIVEL_EDU',
 'ULTCURFAL',
 'MUERTEPORO',
 'SIMUERTEPO',
 'OCUPACION',
 'IDPERTET',
 'CODPRES',
 'CODPTORE',
 'CODMUNRE',
 'AREA_RES',
 'SEG_SOCIAL',
 'IDADMISALUD',
 'P_PMAN_IRIS',
 'CONS_EXP',
 'MU_PARTO',
 'T_PARTO',
 'TIPO_EMB',
 'T_GES',
 'T_GES_AGRU_CIE',
 'PESO_NAC',
 'EDAD_MADRE',
 'N_HIJOSV',
 'N_HIJOSM',
 'EST_CIVM',
 'NIV_EDUM',
 'ULTCURMAD',
 'EMB_FAL',
 'EMB_SEM',
 'EMB_MES',
 'CODOCUR',
 'CODMUNOC',
 'C_MUERTE',
 'C_MUERTEB',
 'C_MUERTEC',
 'C_MUERTED',
 'C_MUERTEE',
 'ASIS_MED',
 'CAUSA_MULT',
 'C_BAS1',
 'CAUSA_667',
 'IDPROFCER',
 'CAU_HOMOL']

In [22]:
########################################################
#Homologar departamento y municipio de defunción #
########################################################
print("\n3. HOMOLOGANDO UBICACIÓN DE DEFUNCIÓN Y OCURRENCIA...")
print(f"   Registros iniciales: {len(df_defun):,}")

# Preparar códigos
df_defun['COD_DPTO'] = df_defun['COD_DPTO'].astype(str).str.zfill(2)
df_defun['COD_MUNIC'] = df_defun['COD_MUNIC'].astype(str).str.zfill(3)

df_defun['CODOCUR'] = pd.to_numeric(df_defun['CODOCUR'], errors='coerce').astype('Int64')
df_defun['CODMUNOC'] = pd.to_numeric(df_defun['CODMUNOC'], errors='coerce').astype('Int64')

# Convertir a string con ceros a la izquierda
df_defun['CODOCUR'] = df_defun['CODOCUR'].astype(str).str.replace('<NA>', 'nan').str.split('.').str[0].str.zfill(2)
df_defun['CODMUNOC'] = df_defun['CODMUNOC'].astype(str).str.replace('<NA>', 'nan').str.split('.').str[0].str.zfill(3)

# Corregir código antiguo 83 -> 18 (Caquetá)
df_defun.loc[df_defun['COD_DPTO'] == '83', 'COD_DPTO'] = '18'
df_defun.loc[df_defun['CODOCUR'] == '83', 'CODOCUR'] = '18'

print(f"   Ejemplo códigos DEFUNCIÓN: DPTO={df_defun['COD_DPTO'].iloc[0]}, MUN={df_defun['COD_MUNIC'].iloc[0]}")
print(f"   Ejemplo códigos OCURRENCIA: DPTO={df_defun['CODOCUR'].iloc[0]}, MUN={df_defun['CODMUNOC'].iloc[0]}")

# Asegurar que divipola_merge no tiene duplicados
divipola_unico = divipola_merge.drop_duplicates(subset=['COD_DPTO', 'COD_MUNIC'])
print(f"   Códigos únicos en DIVIPOLA: {len(divipola_unico):,}")

# =============================================
# HOMOLOGAR LUGAR DE DEFUNCIÓN
# =============================================
print("\n   --- HOMOLOGANDO LUGAR DE DEFUNCIÓN ---")
df_defun = df_defun.merge(
    divipola_unico,
    on=['COD_DPTO', 'COD_MUNIC'],
    how='left',
    suffixes=('', '_DEFUNCION')
)
print(f"   Registros después del merge: {len(df_defun):,}")

# Completar nulos con cabeceras municipales
cabeceras = divipola_unico[divipola_unico['COD_MUNIC'] == '001'][
    ['COD_DPTO', 'NOMBRE_DEPARTAMENTO', 'NOMBRE_MUNICIPIO']
].drop_duplicates(subset=['COD_DPTO']).rename(columns={
    'NOMBRE_DEPARTAMENTO': 'DEPT_CAB_DEF',
    'NOMBRE_MUNICIPIO': 'MUNIC_CAB_DEF'
})

print(f"   Cabeceras municipales: {len(cabeceras):,}")

df_defun = df_defun.merge(cabeceras, on='COD_DPTO', how='left')
print(f"   Registros después de cabeceras: {len(df_defun):,}")

mask_def = df_defun['NOMBRE_DEPARTAMENTO'].isna()
print(f"   Registros con NOMBRE_DEPARTAMENTO nulo: {mask_def.sum():,}")

df_defun.loc[mask_def, 'NOMBRE_DEPARTAMENTO'] = df_defun.loc[mask_def, 'DEPT_CAB_DEF']
df_defun.loc[mask_def, 'NOMBRE_MUNICIPIO'] = df_defun.loc[mask_def, 'MUNIC_CAB_DEF']

# Marcar los que no se encontraron
df_defun['NOMBRE_DEPARTAMENTO'].fillna('NO REGISTRA', inplace=True)
df_defun['NOMBRE_MUNICIPIO'].fillna('NO REGISTRA', inplace=True)

df_defun.drop(['DEPT_CAB_DEF', 'MUNIC_CAB_DEF'], axis=1, inplace=True, errors='ignore')

coincidencias_def = (df_defun['NOMBRE_DEPARTAMENTO'] != 'NO REGISTRA').sum()
print(f"   ✅ Coincidencias DEFUNCIÓN: {coincidencias_def:,} ({coincidencias_def/len(df_defun)*100:.2f}%)")

# =============================================
# HOMOLOGAR LUGAR DE OCURRENCIA
# =============================================
print("\n   --- HOMOLOGANDO LUGAR DE OCURRENCIA ---")
df_defun = df_defun.merge(
    divipola_unico,
    left_on=['CODOCUR', 'CODMUNOC'],
    right_on=['COD_DPTO', 'COD_MUNIC'],
    how='left',
    suffixes=('', '_OCURRENCIA')
)

# Renombrar las columnas del merge
df_defun.rename(columns={
    'NOMBRE_DEPARTAMENTO_OCURRENCIA': 'NOMBRE_DEPARTAMENTO_OCURRENCIA',
    'NOMBRE_MUNICIPIO_OCURRENCIA': 'NOMBRE_MUNICIPIO_OCURRENCIA'
}, inplace=True)

print(f"   Registros después del merge: {len(df_defun):,}")

# Completar nulos con cabeceras municipales
cabeceras_ocur = divipola_unico[divipola_unico['COD_MUNIC'] == '001'][
    ['COD_DPTO', 'NOMBRE_DEPARTAMENTO', 'NOMBRE_MUNICIPIO']
].drop_duplicates(subset=['COD_DPTO']).rename(columns={
    'NOMBRE_DEPARTAMENTO': 'DEPT_CAB_OCUR',
    'NOMBRE_MUNICIPIO': 'MUNIC_CAB_OCUR'
})

df_defun = df_defun.merge(
    cabeceras_ocur,
    left_on='CODOCUR',
    right_on='COD_DPTO',
    how='left',
    suffixes=('', '_y')
)
print(f"   Registros después de cabeceras: {len(df_defun):,}")

mask_ocur = df_defun['NOMBRE_DEPARTAMENTO_OCURRENCIA'].isna()
print(f"   Registros con NOMBRE_DEPARTAMENTO_OCURRENCIA nulo: {mask_ocur.sum():,}")

df_defun.loc[mask_ocur, 'NOMBRE_DEPARTAMENTO_OCURRENCIA'] = df_defun.loc[mask_ocur, 'DEPT_CAB_OCUR']
df_defun.loc[mask_ocur, 'NOMBRE_MUNICIPIO_OCURRENCIA'] = df_defun.loc[mask_ocur, 'MUNIC_CAB_OCUR']

# Marcar los que no se encontraron
df_defun['NOMBRE_DEPARTAMENTO_OCURRENCIA'].fillna('NO REGISTRA', inplace=True)
df_defun['NOMBRE_MUNICIPIO_OCURRENCIA'].fillna('NO REGISTRA', inplace=True)

# Limpiar columnas temporales
cols_drop = [
    'DEPT_CAB_OCUR', 'MUNIC_CAB_OCUR', 
    'COD_DPTO_OCURRENCIA', 'COD_MUNIC_OCURRENCIA',
    'COD_DPTO_y'
]
df_defun.drop([c for c in cols_drop if c in df_defun.columns], axis=1, inplace=True, errors='ignore')

coincidencias_ocur = (df_defun['NOMBRE_DEPARTAMENTO_OCURRENCIA'] != 'NO REGISTRA').sum()
print(f"   ✅ Coincidencias OCURRENCIA: {coincidencias_ocur:,} ({coincidencias_ocur/len(df_defun)*100:.2f}%)")

print(f"\n   📊 RESUMEN FINAL:")
print(f"   Total registros: {len(df_defun):,}")
print(f"   Lugar de DEFUNCIÓN homologado: {coincidencias_def:,}")
print(f"   Lugar de OCURRENCIA homologado: {coincidencias_ocur:,}")


3. HOMOLOGANDO UBICACIÓN DE DEFUNCIÓN Y OCURRENCIA...
   Registros iniciales: 363,089
   Ejemplo códigos DEFUNCIÓN: DPTO=54, MUN=001
   Ejemplo códigos OCURRENCIA: DPTO=nan, MUN=nan
   Códigos únicos en DIVIPOLA: 1,122

   --- HOMOLOGANDO LUGAR DE DEFUNCIÓN ---
   Registros después del merge: 363,089
   Cabeceras municipales: 33
   Registros después de cabeceras: 363,089
   Registros con NOMBRE_DEPARTAMENTO nulo: 2
   ✅ Coincidencias DEFUNCIÓN: 363,089 (100.00%)

   --- HOMOLOGANDO LUGAR DE OCURRENCIA ---
   Registros después del merge: 363,089
   Registros después de cabeceras: 363,089
   Registros con NOMBRE_DEPARTAMENTO_OCURRENCIA nulo: 332,037
   ✅ Coincidencias OCURRENCIA: 31,053 (8.55%)

   📊 RESUMEN FINAL:
   Total registros: 363,089
   Lugar de DEFUNCIÓN homologado: 363,089
   Lugar de OCURRENCIA homologado: 31,053


In [23]:
######################################
# Homologar lugar de residencia #
######################################

print("\n4. HOMOLOGANDO LUGAR DE RESIDENCIA...")
print(f"   Registros antes: {len(df_defun):,}")
df_defun = homologar_departamentos_residencia(df_defun, divipola_merge)
print(f"   Registros después: {len(df_defun):,}")
print(f"   Coincidencias: {(df_defun['NOMBRE_DEPARTAMENTO_RESIDENCIA'] != 'NO REGISTRA').sum():,}")


4. HOMOLOGANDO LUGAR DE RESIDENCIA...
   Registros antes: 363,089
   Registros después del merge: 363,089
   Registros después de cabeceras: 363,089
   Registros después: 363,089
   Coincidencias: 361,157


In [24]:
columnas_a_mapear = [
    'A_DEFUN', 'SEXO', 'EST_CIVIL', 'SIT_DEFUN', 'CONS_EXP', 'AREA_RES', 'P_PMAN_IRIS',
    'TIPO_DEFUN', 'NIVEL_EDU', 'CODPRES', 'CODPTORE', 'SEG_SOCIAL', 'MU_PARTO', 'T_PARTO',
    'TIPO_EMB', 'T_GES', 'PESO_NAC', 'EDAD_MADRE', 'N_HIJOSV', 'N_HIJOSM', 'EST_CIVM',
    'NIV_EDUM', 'EMB_FAL', 'EMB_SEM', 'EMB_MES', 'MAN_MUER', 'CODOCUR', 'C_MUERTE',
    'ASIS_MED', 'IDADMISALUD', 'T_GES_AGRU_CIE', 'IDPROFCER'
]

for col in columnas_a_mapear:
    if col in df_defun.columns:
        df_defun[col] = df_defun[col].apply(lambda x: str(int(x)) if pd.notna(x) and isinstance(x, (float,int)) else str(x) if pd.notna(x) else None)


In [25]:
df_defun.SEXO

0         1
1         1
2         1
3         2
4         1
         ..
363084    1
363085    1
363086    1
363087    1
363088    1
Name: SEXO, Length: 363089, dtype: object

In [ ]:
##################################
# Crear campos descriptivos #
##################################

print("\n5. CREANDO CAMPOS DESCRIPTIVOS...")
mapeos = crear_mapeo_descripciones()

df_defun['A_DEFUN_DESC'] = df_defun['A_DEFUN'].map(mapeos['A_DEFUN'])                           #1
df_defun['SEXO_DESC'] = df_defun['SEXO'].map(mapeos['SEXO'])                                    #2
df_defun['EST_CIVIL_DESC'] = df_defun['EST_CIVIL'].map(mapeos['EST_CIVIL'])                     #3
df_defun['SIT_DEFUN_DESC'] = df_defun['SIT_DEFUN'].map(mapeos['SIT_DEFUN'])                     #4
df_defun['CONS_EXP_DESC'] = df_defun['CONS_EXP'].map(mapeos['CONS_EXP'])                        #5 
df_defun['AREA_RES_DESC'] = df_defun['AREA_RES'].map(mapeos['AREA_RES'])                        #6      
df_defun['PMAN_MUER_DESC'] = df_defun['P_PMAN_IRIS'].map(mapeos['P_PMAN_IRIS'])                 #7 
df_defun['TIPO_DEFUN_DESC'] = df_defun['TIPO_DEFUN'].map(mapeos['TIPO_DEFUN'])                  #8
df_defun['NIVEL_EDU_DESC'] = df_defun['NIVEL_EDU'].map(mapeos['NIVEL_EDU'])                     #9    
df_defun['CODPRES_DESC'] = df_defun['CODPRES'].map(mapeos['CODPRES'])                           #10
df_defun['CODPTORE_DESC'] = df_defun['CODPTORE'].map(mapeos['CODPTORE'])                        #11
df_defun['SEG_SOCIAL_DESC'] = df_defun['SEG_SOCIAL'].map(mapeos['SEG_SOCIAL'])                  #12
df_defun['MU_PARTO_DESC'] = df_defun['MU_PARTO'].map(mapeos['MU_PARTO'])                        #13
df_defun['T_PARTO_DESC'] = df_defun['T_PARTO'].map(mapeos['T_PARTO'])                           #14
df_defun['TIPO_EMB_DESC'] = df_defun['TIPO_EMB'].map(mapeos['TIPO_EMB'])                        #15
df_defun['T_GES_DESC'] = df_defun['T_GES'].map(mapeos['T_GES'])                                 #16    
df_defun['PESO_NAC_DESC'] = df_defun['PESO_NAC'].map(mapeos['PESO_NAC'])                        #17
df_defun['EDAD_MADRE_DESC'] = df_defun['EDAD_MADRE'].map(mapeos['EDAD_MADRE'])                  #18    
df_defun['N_HIJOSV_DESC'] = df_defun['N_HIJOSV'].map(mapeos['N_HIJOSV'])                        #19
df_defun['N_HIJOSM_DESC'] = df_defun['N_HIJOSM'].map(mapeos['N_HIJOSM'])                        #20
df_defun['EST_CIVM_DESC'] = df_defun['EST_CIVM'].map(mapeos['EST_CIVM'])                        #21
df_defun['NIV_EDUM_DESC'] = df_defun['NIV_EDUM'].map(mapeos['NIV_EDUM'])                        #22
df_defun['EMB_FAL_DESC'] = df_defun['EMB_FAL'].map(mapeos['EMB_FAL'])                           #23
df_defun['EMB_SEM_DESC'] = df_defun['EMB_SEM'].map(mapeos['EMB_SEM'])                           #24
df_defun['EMB_MES_DESC'] = df_defun['EMB_MES'].map(mapeos['EMB_MES'])                           #25
df_defun['CODOCUR_DESC'] = df_defun['CODOCUR'].map(mapeos['CODOCUR'])                           #26
df_defun['C_MUERTE_DESC'] = df_defun['C_MUERTE'].map(mapeos['C_MUERTE'])                        #27
df_defun['ASIS_MED_DESC'] = df_defun['ASIS_MED'].map(mapeos['ASIS_MED'])                        #28
df_defun['IDADMISALUD_DESC'] = df_defun['IDADMISALUD'].map(mapeos['IDADMISALUD'])               #29
df_defun['T_GES_AGRU_CIE_DESC'] = df_defun['T_GES_AGRU_CIE'].map(mapeos['T_GES_AGRU_CIE'])      #30
df_defun['IDPROFCER_DESC'] = df_defun['IDPROFCER'].map(mapeos['IDPROFCER'])                     #31

df_defun['GRU_ED1_STR'] = df_defun['GRU_ED1'].astype(str).str.zfill(2)
df_defun['GRU_ED1_DESC'] = df_defun['GRU_ED1_STR'].map(mapeos['GRU_ED1'])

df_defun['GRU_ED2_STR'] = df_defun['GRU_ED2'].astype(str).str.zfill(2)
df_defun['GRU_ED2_DESC'] = df_defun['GRU_ED2_STR'].map(mapeos['GRU_ED2'])

campos = [
    "A_DEFUN_DESC", "SEXO_DESC", "EST_CIVIL_DESC", "SIT_DEFUN_DESC",
    "GRU_ED1_DESC", "GRU_ED2_DESC", "CONS_EXP_DESC", "AREA_RES_DESC",
    "PERMAN_MUN_DESC", "PMAN_MUER_DESC", "TIPO_DEFUN_DESC"
]

print("Campos creados:")
for campo in campos:
    print(f"• {campo}")


5. CREANDO CAMPOS DESCRIPTIVOS...
✓ Países cargados usando ISO 3166-1 (249 países)
Campos creados:
• A_DEFUN_DESC
• SEXO_DESC
• EST_CIVIL_DESC
• SIT_DEFUN_DESC
• GRU_ED1_DESC
• GRU_ED2_DESC
• CONS_EXP_DESC
• AREA_RES_DESC
• PERMAN_MUN_DESC
• PMAN_MUER_DESC
• TIPO_DEFUN_DESC


In [29]:
#################################################
# Homologar causa básica de muerte (CIE-10) #
#################################################

print("\n6. HOMOLOGANDO CAUSA BÁSICA (CIE-10)...")
cie = pd.read_excel("data/raw/Referenciales/CIE_9_10.xls")
cie['CIE10'] = cie['CIE10'].astype(str)

# Eliminar duplicados en CIE
cie_unico = cie.drop_duplicates(subset=['CIE10'])
cie_dict = dict(zip(cie_unico['CIE10'], cie_unico['LITERAL9']))

print("   Creando diccionario CIE-9...")
print(f"   Total códigos CIE-9 únicos: {len(cie_dict):,}")
print(f"   Registros antes del merge: {len(df_defun):,}")

# Merge directo primero
print("   Realizando cruce exacto...")
df_defun = df_defun.merge(cie_unico[['CIE10', 'LITERAL9']], 
                          left_on='C_BAS1', right_on='CIE10', how='left')
df_defun.rename(columns={'LITERAL9': 'C_BAS1_DESC'}, inplace=True)
df_defun.drop('CIE10', axis=1, inplace=True, errors='ignore')

print(f"   Registros después del merge: {len(df_defun):,}")

coincidencias_exactas = df_defun['C_BAS1_DESC'].notna().sum()
print(f"   Coincidencias exactas: {coincidencias_exactas:,} ({coincidencias_exactas/len(df_defun)*100:.2f}%)")

# Búsqueda optimizada para los restantes
sin_coinc = df_defun['C_BAS1_DESC'].isna()
if sin_coinc.sum() > 0:
    print(f"   Buscando con variaciones: {sin_coinc.sum():,} registros...")
    
    # Aplicar búsqueda vectorizada
    tqdm.pandas(desc="   Procesando")
    df_defun.loc[sin_coinc, 'C_BAS1_DESC'] = df_defun.loc[sin_coinc, 'C_BAS1'].progress_apply(
        lambda x: buscar_cie10_optimizado(x, cie_dict)
    )
    
    nuevas_coinc = df_defun.loc[sin_coinc, 'C_BAS1_DESC'].notna().sum()
    print(f"   Nuevas coincidencias: {nuevas_coinc:,}")

total_coinc = df_defun['C_BAS1_DESC'].notna().sum()
print(f"   Total coincidencias: {total_coinc:,} ({total_coinc/len(df_defun)*100:.2f}%)")


6. HOMOLOGANDO CAUSA BÁSICA (CIE-10)...
   Creando diccionario CIE-9...
   Total códigos CIE-9 únicos: 12,421
   Registros antes del merge: 363,089
   Realizando cruce exacto...
   Registros después del merge: 363,089
   Coincidencias exactas: 265,526 (73.13%)
   Buscando con variaciones: 97,563 registros...


   Procesando: 100%|██████████| 97563/97563 [00:00<00:00, 567452.72it/s]


   Nuevas coincidencias: 5,030
   Total coincidencias: 270,556 (74.52%)


In [30]:
############################################
#  Homologar con Lista 105 de Colombia #
############################################

print("\n7. HOMOLOGANDO CON LISTA 105 DE COLOMBIA...")
try:
    # Leer la Lista 105
    ruta_lista105 = "data/raw/Referenciales/Lista_105_Colombia_CIE9-y-CIE10.xls"
    df_lista105 = leer_lista_105(ruta_lista105)
    
    # Mostrar muestra de la Lista 105
    print(f"\n   Muestra de Lista 105 cargada:")
    print(df_lista105[['No_Lista', 'Causa']].head(5))
    
    # Homologar - crea el campo CAU_HOMOL_DESC
    df_defun = homologar_causa_lista105(df_defun, df_lista105)
    
    print(f"\n   ✅ Campo CAU_HOMOL_DESC creado exitosamente")
    
except Exception as e:
    print(f"\n   ⚠️ Error al cargar Lista 105: {e}")
    print(f"   Se continuará sin este campo")


7. HOMOLOGANDO CON LISTA 105 DE COLOMBIA...
Leyendo archivo: data/raw/Referenciales/Lista_105_Colombia_CIE9-y-CIE10.xls
Columnas detectadas: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3']
Datos encontrados a partir de la fila: 6
Columnas identificadas:
  - Número: None
  - Causa: None
  - CIE-10: None
  - CIE-9: None
Usando posición de columnas (A, B, C, D)

✅ Archivo procesado exitosamente
   Total de causas: 105
   Rango de listas: 1 - 105

   Muestra de Lista 105 cargada:
   No_Lista                                              Causa
1         1              Enfermedades infecciosas intestinales
2         2                            Tuberculosis y secuelas
3         3  Ciertas enfermedades transmitidas por vectores...
4         4             Ciertas enfermedades inmunoprevenibles
5         5                       Septicemia, excepto neonatal

Homologando CAU_HOMOL con Lista 105 de Colombia...
   Campo a cruzar: CAU_HOMOL
   Campo a crear: CAU_HOMOL_DESC
   Total códigos 

In [31]:
##########################################
# HOMOLOGAR CAUSA LISTA 6/66 Y 6/67 #
##########################################

print("\n9. HOMOLOGANDO CON LISTAS 6/66 Y 6/67...")

try:
    # Cargar el archivo de referencia
    df_ref = cargar_comparativo()
    print(f"   ✅ Archivo de referencia cargado: {len(df_ref):,} registros")
    
    # ------------------------------------
    # CRUCE 1: CAUSA_666 con lista 666
    # ------------------------------------
    if 'CAUSA_666' in df_defun.columns:
        print(f"\n   🔄 Homologando CAUSA_666 con Lista 6/66...")
        df_defun = anexar_desc(
            df_defun=df_defun,
            df_ref=df_ref,
            columna_origen='CAUSA_666',
            lista_referencia='666'
        )
        
        coincidencias = df_defun['CAUSA_666_DESC'].notna().sum()
        porcentaje = (coincidencias / len(df_defun)) * 100
        print(f"   ✅ Campo creado: CAUSA_666_DESC")
        print(f"   📊 Coincidencias: {coincidencias:,} ({porcentaje:.2f}%)")
    
    # ------------------------------------
    # CRUCE 2: CAUSA_667 con lista 667
    # ------------------------------------
    if 'CAUSA_667' in df_defun.columns:
        print(f"\n   🔄 Homologando CAUSA_667 con Lista 6/67...")
        df_defun = anexar_desc(
            df_defun=df_defun,
            df_ref=df_ref,
            columna_origen='CAUSA_667',
            lista_referencia='667'
        )
        
        coincidencias = df_defun['CAUSA_667_DESC'].notna().sum()
        porcentaje = (coincidencias / len(df_defun)) * 100
        print(f"   ✅ Campo creado: CAUSA_667_DESC")
        print(f"   📊 Coincidencias: {coincidencias:,} ({porcentaje:.2f}%)")
    
    print("\n   ✅ Todas las homologaciones completadas")
    
except Exception as e:
    print(f"\n   ❌ Error: {e}")
    import traceback
    traceback.print_exc()


9. HOMOLOGANDO CON LISTAS 6/66 Y 6/67...
   ✅ Archivo de referencia cargado: 67 registros

   🔄 Homologando CAUSA_667 con Lista 6/67...
   ✅ Campo creado: CAUSA_667_DESC
   📊 Coincidencias: 340,723 (93.84%)

   ✅ Todas las homologaciones completadas


In [32]:
#####################################
# Mostrar DataFrame procesado #
#####################################

pd.set_option('display.max_columns', None) 
display(df_defun.head())


,COD_DPTO,COD_MUNIC,A_DEFUN,SIT_DEFUN,OTRSITIODE,TIPO_DEFUN,ANO,MES,HORA,MINUTOS,SEXO,EST_CIVIL,GRU_ED1,GRU_ED2,NIVEL_EDU,ULTCURFAL,MUERTEPORO,SIMUERTEPO,OCUPACION,IDPERTET,CODPRES,CODPTORE,CODMUNRE,AREA_RES,SEG_SOCIAL,IDADMISALUD,P_PMAN_IRIS,CONS_EXP,MU_PARTO,T_PARTO,TIPO_EMB,T_GES,T_GES_AGRU_CIE,PESO_NAC,EDAD_MADRE,N_HIJOSV,N_HIJOSM,EST_CIVM,NIV_EDUM,ULTCURMAD,EMB_FAL,EMB_SEM,EMB_MES,CODOCUR,CODMUNOC,C_MUERTE,C_MUERTEB,C_MUERTEC,C_MUERTED,C_MUERTEE,ASIS_MED,CAUSA_MULT,C_BAS1,CAUSA_667,IDPROFCER,CAU_HOMOL,NOMBRE_DEPARTAMENTO,NOMBRE_MUNICIPIO,NOMBRE_DEPARTAMENTO_OCURRENCIA,NOMBRE_MUNICIPIO_OCURRENCIA,NOMBRE_DEPARTAMENTO_RESIDENCIA,NOMBRE_MUNICIPIO_RESIDENCIA,A_DEFUN_DESC,SEXO_DESC,EST_CIVIL_DESC,SIT_DEFUN_DESC,CONS_EXP_DESC,AREA_RES_DESC,PMAN_MUER_DESC,TIPO_DEFUN_DESC,NIVEL_EDU_DESC,CODPRES_DESC,CODPTORE_DESC,SEG_SOCIAL_DESC,MU_PARTO_DESC,T_PARTO_DESC,TIPO_EMB_DESC,T_GES_DESC,PESO_NAC_DESC,EDAD_MADRE_DESC,N_HIJOSV_DESC,N_HIJOSM_DESC,EST_CIVM_DESC,NIV_EDUM_DESC,EMB_FAL_DESC,EMB_SEM_DESC,EMB_MES_DESC,CODOCUR_DESC,C_MUERTE_DESC,ASIS_MED_DESC,IDADMISALUD_DESC,T_GES_AGRU_CIE_DESC,IDPROFCER_DESC,GRU_ED1_STR,GRU_ED1_DESC,GRU_ED2_STR,GRU_ED2_DESC,C_BAS1_DESC,CAU_HOMOL_DESC,CAUSA_667_DESC
0,54,001,1,1,NaN,2,2021,5,3.0,25.0,1,9,21,6,99,99,2.0,NaN,SIN INFORMACION,6,170,54,001,1,2,2,0,2,None,None,None,None,None,None,None,None,None,None,None,NaN,None,None,None,nan,nan,None,1.0,NaN,NaN,NaN,1,J960/J129/U071*E149,U071,109,1,10,NORTE DE SANTANDER,SAN JOSÉ DE CÚCUTA,NO REGISTRA,NO REGISTRA,NORTE DE SANTANDER,SAN JOSÉ DE CÚCUTA,CABECERA MUNICIPAL,MASCULINO,SIN INFORMACIÓN,HOSPITAL/CLÍNICA,MÉDICO NO TRATANTE,CABECERA MUNICIPAL,NaN,DEFUNCIÓN NO FETAL,SIN INFORMACIÓN,COLOMBIA,NORTE DE SANTANDER,SUBSIDIADO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SÍ,ENTIDAD PROMOTORA DE SALUD - SUBSIDIADO,NaN,MÉDICO,21,DE 65 A 69 AÑOS,06,DE 65 Y MÁS AÑOS,None,Todas las demás enfermedades infecciosas y par...,INFECCIONES RESPIRATORIAS AGUDAS
1,54,001,1,1,NaN,2,2021,12,6.0,5.0,1,6,21,6,99,99,2.0,NaN,HOGAR,6,170,54,405,1,3,5,0,2,None,None,None,None,None,None,None,None,None,None,None,NaN,None,None,None,nan,nan,None,1.0,NaN,NaN,NaN,1,R570/I269/I742*I10,I742,309,1,58,NORTE DE SANTANDER,SAN JOSÉ DE CÚCUTA,NO REGISTRA,NO REGISTRA,NORTE DE SANTANDER,LOS PATIOS,CABECERA MUNICIPAL,MASCULINO,ESTABA CASADO(A),HOSPITAL/CLÍNICA,MÉDICO NO TRATANTE,CABECERA MUNICIPAL,NaN,DEFUNCIÓN NO FETAL,SIN INFORMACIÓN,COLOMBIA,NORTE DE SANTANDER,EXCEPCIÓN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SÍ,ENTIDAD EXCEPTUADA DE SALUD,NaN,MÉDICO,21,DE 65 A 69 AÑOS,06,DE 65 Y MÁS AÑOS,EMBOLIA Y TROMBOSIS DE ARTERIAS DE LOS MIEMBROS,Enfermedades de los vasos sanguíneos y otras e...,OTRAS DE ENFERMEDADES DEL SISTEMA CIRCULATORIO
2,54,001,1,1,NaN,2,2021,7,19.0,30.0,1,9,18,5,99,99,2.0,NaN,SIN INFORMACION,6,170,54,001,1,2,2,2,2,None,None,None,None,None,None,None,None,None,None,None,NaN,None,None,None,54,001,None,1.0,NaN,NaN,NaN,1,J969/G934 N19/N179/T659 X49*F319,X499,509,1,99,NORTE DE SANTANDER,SAN JOSÉ DE CÚCUTA,NORTE DE SANTANDER,SAN JOSÉ DE CÚCUTA,NORTE DE SANTANDER,SAN JOSÉ DE CÚCUTA,CABECERA MUNICIPAL,MASCULINO,SIN INFORMACIÓN,HOSPITAL/CLÍNICA,MÉDICO NO TRATANTE,CABECERA MUNICIPAL,VIOLENTA,DEFUNCIÓN NO FETAL,SIN INFORMACIÓN,COLOMBIA,NORTE DE SANTANDER,SUBSIDIADO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NORTE DE SANTANDER,NaN,SÍ,ENTIDAD PROMOTORA DE SALUD - SUBSIDIADO,NaN,MÉDICO,18,DE 50 A 54 AÑOS,05,DE 45 A 64 AÑOS,ENVENENAMIENTO ACC.OTRAS DROGAS:ACTUAN METABOL...,"Envenenamiento accidental por, y exposición a ...",NaN
3,63,001,1,3,NaN,2,2021,2,21.0,30.0,2,4,26,6,2,5,2.0,NaN,NaN,6,170,63,001,1,1,1,0,2,None,None,None,None,None,None,None,None,None,None,None,NaN,None,None,None,nan,nan,None,1.0,NaN,1.0,NaN,1,I509/R688/R590,I509,306,1,54,QUINDÍO,ARMENIA,NO REGISTRA,NO REGISTRA,QUINDÍO,ARMENIA,CABECERA MUNICIPAL,FEMENINO,ESTABA VIUDO(A),CASA/DOMICILIO,MÉDICO NO TRATANTE,CABECERA MUNICIPAL,NaN,DEFUNCIÓN NO FETAL,BÁSICA PRIMARIA,COLOMBIA,QUINDÍO,CONTRIBUTIVO,NaN,Na

In [33]:
#####################################
# Guardar DataFrame procesado #
#####################################

print("\n8. GENERANDO ARCHIVO PROCESADO...")
try:
    # Nombre descriptivo del archivo
    nombre_archivo = f"defunciones_{2021}_procesado"
    
    # Guardar
    ruta_guardada = guardar_dataframe_procesado(
        df_defun, 
        nombre_archivo=nombre_archivo,
        ruta_carpeta="data/processed/"
    )
    
    print(f"\n   🎉 DataFrame guardado correctamente")
    
except Exception as e:
    print(f"\n   ❌ Error al guardar archivo: {e}")
    print(f"   El proceso continuó pero no se guardó el archivo")


8. GENERANDO ARCHIVO PROCESADO...
   ✨ Creando nuevo archivo: data\processed\defunciones_2021_procesado.parquet
   ✅ Archivo guardado exitosamente
   📊 Registros: 363,089
   📁 Tamaño: 42.61 MB
   📂 Ruta: data\processed\defunciones_2021_procesado.parquet

   💡 Para leer este archivo después, usa:
      df = pd.read_parquet('data\processed\defunciones_2021_procesado.parquet')

   🎉 DataFrame guardado correctamente
